In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_PATH = "data/IMDB Dataset.csv"
OUTPUT_PATH = "output/cleaned_reviews"

spark = (
    SparkSession.builder.appName("pyspark-movie-reviews")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.1.2'

In [2]:
df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(DATA_PATH)
)
print(f"Rows loaded: {df.count()}")

Rows loaded: 50000


In [3]:
df.printSchema()

root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)



In [4]:
df.show(5, truncate=80)

+--------------------------------------------------------------------------------+---------+
|                                                                          review|sentiment|
+--------------------------------------------------------------------------------+---------+
|One of the other reviewers has mentioned that after watching just 1 Oz episod...| positive|
|A wonderful little production. <br /><br />The filming technique is very unas...| positive|
|I thought this was a wonderful way to spend time on a too hot summer weekend,...| positive|
|Basically there's a family where a little boy (Jake) thinks there's a zombie ...| negative|
|Petter Mattei's "Love in the Time of Money" is a visually stunning film to wa...| positive|
+--------------------------------------------------------------------------------+---------+
only showing top 5 rows


In [5]:
clean_df = df.filter(F.col("review").isNotNull() & (F.trim(F.col("review")) != ""))
print(f"Rows before cleaning: {df.count()}")
print(f"Rows after cleaning:  {clean_df.count()}")

Rows before cleaning: 50000


Rows after cleaning:  50000


In [6]:
print(f"Total reviews: {clean_df.count()}")

Total reviews: 50000


In [7]:
clean_df.groupBy("sentiment").count().orderBy("sentiment").show()

+---------+-----+
|sentiment|count|
+---------+-----+
| negative|25000|
| positive|25000|
+---------+-----+



In [8]:
transformed_df = clean_df.withColumn("review_length", F.length("review"))
transformed_df.select("sentiment", "review_length").show(5)

+---------+-------------+
|sentiment|review_length|
+---------+-------------+
| positive|         1761|
| positive|          998|
| positive|          926|
| negative|          748|
| positive|         1317|
+---------+-------------+
only showing top 5 rows


In [9]:
long_df = transformed_df.filter(F.col("review_length") > 500)
print(f"Reviews longer than 500 characters: {long_df.count()}")
long_df.select("sentiment", "review_length", "review").show(5, truncate=80)

Reviews longer than 500 characters: 45017
+---------+-------------+--------------------------------------------------------------------------------+
|sentiment|review_length|                                                                          review|
+---------+-------------+--------------------------------------------------------------------------------+
| positive|         1761|One of the other reviewers has mentioned that after watching just 1 Oz episod...|
| positive|          998|A wonderful little production. <br /><br />The filming technique is very unas...|
| positive|          926|I thought this was a wonderful way to spend time on a too hot summer weekend,...|
| negative|          748|Basically there's a family where a little boy (Jake) thinks there's a zombie ...|
| positive|         1317|Petter Mattei's "Love in the Time of Money" is a visually stunning film to wa...|
+---------+-------------+-----------------------------------------------------------------------------

In [10]:
(
    transformed_df.coalesce(1).write.mode("overwrite")
    .option("header", True)
    .option("escape", '"')
    .csv(OUTPUT_PATH)
)
print(f"Saved cleaned and transformed data to {OUTPUT_PATH}/")

Saved cleaned and transformed data to output/cleaned_reviews/


In [11]:
spark.stop()